# DSC 442 Final Exam
In this exam, you will use the steel industry electricity dataset to predict `Usage_kWh` using a `RandomForestRegressor`.

**About The Dataset**

Steel plants use a large amount of electricity because many machines and industrial systems run throughout the day. The goal is to build a machine learning model that can predict how much electricity was used during a given time period.

Each row in the dataset represents one recorded time period. The main value we want to predict is:

`Usage_kWh`

This means the amount of electricity used, measured in kilowatt-hours.

**Column Descriptions**

`date`  
The date and time when the measurement was recorded.

`Usage_kWh`  
The electricity usage in kilowatt-hours. This is the target column we are trying to predict.

`Lagging_Current_Reactive.Power_kVarh`  
A measurement related to reactive electrical power in the system.

`Leading_Current_Reactive_Power_kVarh`  
Another measurement related to reactive electrical power.

`CO2(tCO2)`  
The amount of carbon dioxide emissions. This is related to energy usage, so we will be careful about using it as a prediction feature.

`Lagging_Current_Power_Factor`  
A measurement related to how efficiently electricity is being used.

`Leading_Current_Power_Factor`  
Another measurement related to electrical efficiency.

`NSM`  
Number of seconds from midnight. This helps represent the time of day.

`WeekStatus`  
Tells whether the row is from a weekday or weekend.

`Day_of_week`  
Tells which day of the week the measurement was recorded.

`Load_Type`  
Describes the demand level in the plant, such as light load, medium load, or maximum load.

## Goal

Use the available columns to train a machine learning model that predicts `Usage_kWh`.
You will answer the final questions in Canvas. The notebook is here to help you compute those answers.

- Use `random_state=0` whenever the question tells you to use it.
- Do not change the sample size.
- Do not use `CO2(tCO2)` as a modeling feature. It is too directly connected to energy usage.
- Round answers exactly as requested in Canvas.

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.feature_selection import mutual_info_regression


Run this cell first. Do not change the sampling line.


In [ ]:
df = pd.read_csv("Steel_industry_data.csv")

# This makes the exam dataset fixed and deterministic.
# Do not change n or random_state.
exam_df = df.sample(n=8000, random_state=0).reset_index(drop=True)

# Use this to preview the dataset.
exam_df.head()


## Part 1: Load and inspect the dataset

In this part, you will use basic pandas commands to inspect the data.

You are not building a machine learning model yet. You are only learning what is inside the dataset.

Answer the following Canvas questions using `exam_df`.

### Question 1

What is the shape of `exam_df`? Enter the number of rows

For example, if there are 100 rows and 5 columns, enter:

```text
100
```
### Question 2

What is the shape of `exam_df`? Enter the number of columns

For example, if there are 100 rows and 5 columns, enter:

```text
5
```
### Question 3

What is the mean of `Usage_kWh` in `exam_df`, rounded to 2 decimal places?

### Question 4

What is the maximum value of `Usage_kWh` in `exam_df`, rounded to 2 decimal places?

### Question 5

How many total missing values are in `exam_df`?

This means the total number of missing cells in the whole DataFrame, not just one column.

### Question 6

What is the most common value in the `Load_Type` column?

### Question 7

Look at row 0 of `exam_df`. What is the `Day_of_week` value?

### Question 8

Look at row 0 of `exam_df`. What is the `NSM` value?


In [ ]:
# Use pandas commands to answer Questions 1--7.


# Q1: shape of exam_df. Number of rows.



# Q2: shape of exam_df. Number of columns. 



# Q3: mean Usage_kWh rounded to 2 decimals



# Q4: maximum Usage_kWh rounded to 2 decimals



# Q5: total number of missing values



# Q6: most common Load_Type



# Q7-Q8: row 0 values for Day_of_week and NSM



## Part 2: Preparing features with one-hot encoding

Machine learning models need numerical input. Some columns in our data contain words, such as `WeekStatus`, `Day_of_week`, and `Load_Type`.

One-hot encoding turns a word column into several 0/1 columns.

For example, if a column called `load_type` contains `Light`, `Medium`, and `Heavy`, then one-hot encoding creates columns such as:

```text
load_type_Light
load_type_Medium
load_type_Heavy
```

The toy example below demonstrates every step you will later do on the real dataset:

1. Start with a small DataFrame.
2. Separate the target column from the feature columns.
3. Identify the categorical columns.
4. Use `pd.get_dummies` to one-hot encode the categorical columns.
5. Check the new columns.


In [ ]:
toy_df = pd.DataFrame({
    "temperature": [70, 75, 80, 72],
    "machine_age": [2, 5, 3, 6],
    "load_type": ["Light", "Medium", "Light", "Heavy"],
    "day_type": ["Weekday", "Weekday", "Weekend", "Weekend"],
    "usage": [10, 18, 12, 25]
})

print("Original toy data:")
display(toy_df)

# Step 1: Separate features X from target y.
# usage is what we are trying to predict, so it becomes y.
toy_X = toy_df.drop("usage", axis=1)
toy_y = toy_df["usage"]

# Step 2: Identify categorical columns.
toy_categorical_cols = ["load_type", "day_type"]

# Step 3: One-hot encode the categorical columns.
toy_X_encoded = pd.get_dummies(
    toy_X,
    columns=toy_categorical_cols,
    drop_first=False
)

print("Toy features after one-hot encoding:")
display(toy_X_encoded)

print("Toy encoded column names:")
print(list(toy_X_encoded.columns))

print("Does toy_X_encoded contain load_type_Medium?")
print("load_type_Medium" in toy_X_encoded.columns)


### Questions 9--10: One-hot encode the real exam data

Now repeat the same idea on the real dataset.

For the basic model, first remove two columns:

- remove `date`, because it is a raw date string and we are not using it in the basic model;
- remove `CO2(tCO2)`, because it is too directly connected to energy usage and would make prediction too easy.

Then:

1. Create `X` by dropping the target column `Usage_kWh`.
2. Create `y` from the target column `Usage_kWh`.
3. One-hot encode the categorical columns:
   - `WeekStatus`
   - `Day_of_week`
   - `Load_Type`

### Question 9

After one-hot encoding, how many columns are in `X_encoded`?

### Question 10

Does `X_encoded` contain a column named exactly:

```text
Load_Type_Medium_Load
```

Enter:

```text
True
```

or

```text
False
```


In [ ]:
# Questions 9--10
# Complete this code by replacing the blanks.

basic_df = exam_df.drop(columns=["date", "CO2(tCO2)"])

# Create X and y.
# X should contain the input features.
# y should contain the target column Usage_kWh.
X = basic_df.drop("____", axis=1)
y = basic_df["____"]

categorical_cols = ["WeekStatus", "Day_of_week", "Load_Type"]

# One-hot encode the categorical columns.
X_encoded = pd.get_dummies(
    X,
    columns=____,
    drop_first=False
)




## Part 3: Train/test split and Random Forest model

A train/test split divides the data into two parts.

The model learns from the training data. Then we test it on data it did not train on.

This helps us estimate whether the model can make useful predictions on new data.

The toy example below demonstrates every step you will later do on the real dataset:

1. Split `toy_X_encoded` and `toy_y` into training and testing sets.
2. Create a `RandomForestRegressor`.
3. Fit the model on the training data.
4. Predict on the test data.
5. Compute the mean absolute error, or MAE.

MAE means the average size of the prediction error. If the MAE is 3.5, then the model is wrong by about 3.5 units on average.


In [ ]:
toy_X_train, toy_X_test, toy_y_train, toy_y_test = train_test_split(
    toy_X_encoded,
    toy_y,
    test_size=0.5,
    random_state=0
)

print("Toy training rows:", toy_X_train.shape[0])
print("Toy testing rows:", toy_X_test.shape[0])

toy_model = RandomForestRegressor(
    n_estimators=10,
    max_depth=3,
    random_state=0
)

toy_model.fit(toy_X_train, toy_y_train)

toy_preds = toy_model.predict(toy_X_test)

toy_mae = mean_absolute_error(toy_y_test, toy_preds)

print("Toy predictions:", toy_preds)
print("Toy MAE:", toy_mae)


### Questions 11--13: Train the basic Random Forest model

Now repeat the same process on the real encoded dataset.

Use:

```python
train_test_split(..., test_size=0.2, random_state=0)
```

Use this model:

```python
RandomForestRegressor(
    n_estimators=50,
    max_depth=8,
    random_state=0
)
```

Then train the model and compute the MAE.

### Question 11

How many rows are in the training set and testing set?

Enter your answer as:

```text
training_rows,testing_rows
```

For example:

```text
80,20
```

### Question 12

What is the basic model MAE, rounded to 2 decimal places?

### Question 13

What are the first five predictions from the basic model on `X_test`?

Enter them rounded to 2 decimal places.

Example format:

```text
12.34,56.78,9.10,11.12,13.14
```


In [ ]:
# Questions 11--13
# Complete this code by replacing the blanks.

X_train, X_test, y_train, y_test = train_test_split(
    ____,
    ____,
    test_size=____,
    random_state=____
)

basic_model = RandomForestRegressor(
    n_estimators=____,
    max_depth=____,
    random_state=____
)

# Fit the model on the training data.
basic_model.fit(____, ____)

# Predict on the testing features.
basic_preds = basic_model.predict(____)

# Compute the MAE by comparing the true testing target to the predictions.
basic_mae = mean_absolute_error(____, ____)


## Part 4: Feature engineering

Feature engineering means creating new useful columns from columns we already have.

The toy example below demonstrates every step you will later do on the real dataset:

1. Convert a date column into a real pandas date.
2. Create an `hour` column from seconds after midnight.
3. Create a `month` column from the date.
4. Add two related numeric columns to create a total.
5. Subtract two related numeric columns to create a gap.
6. Create a 0/1 column for whether the time is a working time.
7. One-hot encode categorical features.
8. Train a Random Forest model again.
9. Compute MAE again.


In [ ]:
toy_eng = pd.DataFrame({
    "date": ["01/01/2018 08:00", "01/01/2018 12:00", "02/01/2018 18:00", "02/01/2018 23:00", "03/01/2018 09:00", "03/01/2018 15:00"],
    "seconds_after_midnight": [28800, 43200, 64800, 82800, 32400, 54000],
    "reactive_a": [2.0, 4.0, 3.5, 5.0, 2.5, 4.5],
    "reactive_b": [1.0, 1.5, 2.0, 1.0, 1.2, 1.8],
    "factor_a": [95, 92, 90, 88, 94, 91],
    "factor_b": [89, 87, 86, 83, 90, 85],
    "load_type": ["Light", "Medium", "Heavy", "Heavy", "Light", "Medium"],
    "usage": [10, 18, 30, 34, 12, 20]
})

toy_eng["date"] = pd.to_datetime(toy_eng["date"], dayfirst=True)

toy_eng["hour"] = toy_eng["seconds_after_midnight"] // 3600
toy_eng["month"] = toy_eng["date"].dt.month
toy_eng["reactive_total"] = toy_eng["reactive_a"] + toy_eng["reactive_b"]
toy_eng["factor_gap"] = toy_eng["factor_a"] - toy_eng["factor_b"]
toy_eng["is_working_time"] = ((toy_eng["hour"] >= 8) & (toy_eng["hour"] <= 17)).astype(int)

print("Toy data after feature engineering:")
display(toy_eng)

toy_eng_model_df = toy_eng.drop(columns=["date"])
toy_eng_X = toy_eng_model_df.drop("usage", axis=1)
toy_eng_y = toy_eng_model_df["usage"]

toy_eng_X_encoded = pd.get_dummies(
    toy_eng_X,
    columns=["load_type"],
    drop_first=False
)

toy_eng_X_train, toy_eng_X_test, toy_eng_y_train, toy_eng_y_test = train_test_split(
    toy_eng_X_encoded,
    toy_eng_y,
    test_size=0.5,
    random_state=0
)

toy_eng_model = RandomForestRegressor(
    n_estimators=10,
    max_depth=3,
    random_state=0
)

toy_eng_model.fit(toy_eng_X_train, toy_eng_y_train)
toy_eng_preds = toy_eng_model.predict(toy_eng_X_test)
toy_eng_mae = mean_absolute_error(toy_eng_y_test, toy_eng_preds)

print("Toy engineered MAE:", toy_eng_mae)


### Questions 14--16: Create engineered features and train an improved model

Now repeat the same feature-engineering process on the real dataset.

Create a copy called `eng_df`.

Then create these new columns:

### `hour`

The column `NSM` means seconds from midnight.

Since there are 3600 seconds in one hour, create:

```python
hour = NSM // 3600
```

### `month`

Convert the `date` column to a pandas datetime, then extract the month.

### `reactive_power_total`

Add:

```text
Lagging_Current_Reactive.Power_kVarh
```

and

```text
Leading_Current_Reactive_Power_kVarh
```

### `power_factor_gap`

Subtract:

```text
Leading_Current_Power_Factor
```

from:

```text
Lagging_Current_Power_Factor
```

### `is_working_time`

Create a 0/1 column.

It should be 1 when the hour is between 8 and 17, inclusive.

It should be 0 otherwise.

After creating these features, train a new Random Forest model using the same settings as before:

```python
RandomForestRegressor(
    n_estimators=50,
    max_depth=8,
    random_state=0
)
```

Use:

```python
train_test_split(..., test_size=0.2, random_state=0)
```

### Question 14

After one-hot encoding the engineered dataset, how many columns are in `X_eng_encoded`?

### Question 15

What is the engineered model MAE, rounded to 2 decimal places?

### Question 16

How much did the engineered model improve over the basic model?

Compute:

```text
basic_mae - engineered_mae
```

Round your answer to 2 decimal places.

If the answer is positive, the engineered model did better. If the answer is negative, the engineered model did worse.


In [ ]:
# Questions 14--16
# Complete this code by replacing the blanks.

eng_df = exam_df.copy()

# Convert date to pandas datetime.
eng_df["date"] = pd.to_datetime(eng_df["date"], dayfirst=True)

# Create the engineered features.
eng_df["hour"] = eng_df["NSM"] // ____
eng_df["month"] = eng_df["date"].dt.____
eng_df["reactive_power_total"] = eng_df["Lagging_Current_Reactive.Power_kVarh"] + eng_df["Leading_Current_Reactive_Power_kVarh"]
eng_df["power_factor_gap"] = eng_df["Lagging_Current_Power_Factor"] - eng_df["Leading_Current_Power_Factor"]
eng_df["is_working_time"] = ((eng_df["hour"] >= ____) & (eng_df["hour"] <= ____)).astype(int)

# Remove date and CO2(tCO2) before modeling.
eng_model_df = eng_df.drop(columns=["date", "CO2(tCO2)"])

# Create X_eng and y_eng.
X_eng = eng_model_df.drop("____", axis=1)
y_eng = eng_model_df["____"]

# One-hot encode the same categorical columns as before.
X_eng_encoded = pd.get_dummies(
    X_eng,
    columns=categorical_cols,
    drop_first=False
)

# Split the engineered data.
X_train_eng, X_test_eng, y_train_eng, y_test_eng = train_test_split(
    ____,
    ____,
    test_size=____,
    random_state=____
)

engineered_model = RandomForestRegressor(
    n_estimators=____,
    max_depth=____,
    random_state=____
)

engineered_model.fit(____, ____)
engineered_preds = engineered_model.predict(____)
engineered_mae = mean_absolute_error(____, ____)


## Part 5: Cross-validation folds

A normal train/test split tests the model once.

Cross-validation tests the model several times.

If `cv=3`, the data is split into 3 parts called folds.

The model trains and tests 3 times:

```text
Run 1: train on folds 2 and 3, test on fold 1
Run 2: train on folds 1 and 3, test on fold 2
Run 3: train on folds 1 and 2, test on fold 3
```

In scikit-learn, we use:

```python
cross_val_score(...)
```

For MAE, scikit-learn uses:

```python
scoring="neg_mean_absolute_error"
```

That gives negative numbers because scikit-learn treats bigger scores as better. Since MAE is an error and smaller is better, scikit-learn stores it as negative.

So we multiply by `-1` to get normal positive MAE values.

The toy example below demonstrates the full process.


In [ ]:
toy_cv_model = RandomForestRegressor(
    n_estimators=10,
    max_depth=3,
    random_state=0
)

toy_cv_scores = cross_val_score(
    toy_cv_model,
    toy_eng_X_encoded,
    toy_eng_y,
    cv=3,
    scoring="neg_mean_absolute_error"
)

toy_cv_mae_scores = -toy_cv_scores
toy_cv_average_mae = toy_cv_mae_scores.mean()

print("Toy negative CV scores from sklearn:")
print(toy_cv_scores)

print("Toy positive MAE scores:")
print(toy_cv_mae_scores)

print("Toy average CV MAE:")
print(toy_cv_average_mae)


### Question 17: Cross-validation on the real engineered features

Now repeat the same cross-validation process on the real engineered dataset.

To keep the exam fast in JupyterLite, use only the first 2000 rows:

```python
cv_X = X_eng_encoded.head(2000)
cv_y = y_eng.head(2000)
```

Use this model:

```python
RandomForestRegressor(
    n_estimators=30,
    max_depth=8,
    random_state=0
)
```

Use:

```python
cv=5
```

and:

```python
scoring="neg_mean_absolute_error"
```

Then convert the scores to positive MAE values and compute the average.

### Question 17

What is the average cross-validation MAE, rounded to 2 decimal places?


In [ ]:
# Question 17
# Complete this code by replacing the blanks.

cv_X = X_eng_encoded.head(____)
cv_y = y_eng.head(____)

cv_model = RandomForestRegressor(
    n_estimators=____,
    max_depth=____,
    random_state=____
)

cv_scores = cross_val_score(
    ____,
    ____,
    ____,
    cv=____,
    scoring="____"
)

cv_mae_scores = -cv_scores
cv_average_mae = cv_mae_scores.mean()


## Part 6: Mutual information and Random Forest feature importance

This part is about identifying which features seem useful.

### Mutual information

Mutual information is a score that tries to measure how much knowing one feature helps us understand the target.

A larger mutual information score means the feature may contain more useful information about the target.

### Random Forest feature importance

A Random Forest can also report feature importance after it is trained.

This tells us which features the trained forest used most strongly when making predictions.

These two methods are related, but they are not the same. They can choose different top features.

The toy example below demonstrates the full process:

1. Compute mutual information scores.
2. Put the scores into a pandas Series.
3. Sort the scores.
4. Find the top mutual information feature.
5. Use the already trained Random Forest model's `feature_importances_`.
6. Sort those scores.
7. Find the top Random Forest feature.


In [ ]:
toy_mi_scores = mutual_info_regression(
    toy_eng_X_encoded,
    toy_eng_y,
    random_state=0
)

toy_mi_series = pd.Series(
    toy_mi_scores,
    index=toy_eng_X_encoded.columns
).sort_values(ascending=False)

print("Toy mutual information scores:")
display(toy_mi_series)

print("Toy top MI feature:")
print(toy_mi_series.index[0])

toy_importance_series = pd.Series(
    toy_eng_model.feature_importances_,
    index=toy_eng_X_encoded.columns
).sort_values(ascending=False)

print("Toy Random Forest feature importances:")
display(toy_importance_series)

print("Toy top Random Forest feature:")
print(toy_importance_series.index[0])


### Questions 18--19: Feature scores on the real engineered dataset

Now repeat the same process on the real engineered dataset.

Use:

```python
mutual_info_regression(X_eng_encoded, y_eng, random_state=0)
```

Then create a pandas Series using `X_eng_encoded.columns` as the index, sort it from largest to smallest, and identify the first index.

Then use:

```python
engineered_model.feature_importances_
```

to find the top Random Forest feature.

### Question 18

Which feature has the highest mutual information score?

Enter the feature name exactly as it appears in the notebook.

### Question 19

Which feature has the highest Random Forest feature importance?

Enter the feature name exactly as it appears in the notebook.


In [ ]:
# Questions 18--19
# Complete this code by replacing the blanks.

mi_scores = mutual_info_regression(
    ____,
    ____,
    random_state=____
)

mi_series = pd.Series(
    mi_scores,
    index=____.columns
).sort_values(ascending=False)

# Random Forest feature importances from the trained engineered_model.
importance_series = pd.Series(
    engineered_model.feature_importances_,
    index=____.columns
).sort_values(ascending=False)

